# GC Application 12 Extractor

Extracts two CSVs from the GC pay application PDF:
1. **`gc_app12_cover.csv`** — G702 cover page fields (one row)
2. **`gc_app12_line_items.csv`** — G703 continuation sheet (one row per line item)

In [30]:
# ── Cell 1 · Install dependencies ────────────────────────────────────────────
%pip install openai pandas tqdm json-repair pymupdf --quiet


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [42]:
# ── Cell 2 · Configuration ────────────────────────────────────────────────────
import os

# ── Azure Document Intelligence ──
DOC_INTEL_ENDPOINT = os.environ.get("DOC_INTEL_ENDPOINT", "")
DOC_INTEL_KEY      = os.environ.get("DOC_INTEL_KEY", "")

# ── Azure OpenAI — GPT-5.4 via Responses API ──
AOAI_ENDPOINT    = os.environ.get("AOAI_ENDPOINT", "")
AOAI_KEY         = os.environ.get("AOAI_KEY", "")
AOAI_DEPLOYMENT  = "gpt-5.4"

# ── Paths ──
BASE_DIR         = r"C:\Users\KR614XU\Downloads\Ishaan"
PDF_PATH         = os.path.join(BASE_DIR, "Test Files", "App 12.pdf")
OCR_CACHE        = os.path.join(BASE_DIR, "gc_app12_ocr_cache.json")
COVER_CSV        = os.path.join(BASE_DIR, "gc_app12_cover_v3.csv")
LINE_ITEMS_CSV   = os.path.join(BASE_DIR, "gc_app12_line_items_v3.csv")

# ── Tuning ──
PAGE_TEXT_LIMIT  = 8000   # chars per page sent to OpenAI

from openai import OpenAI
aoai_client = OpenAI(
    base_url=AOAI_ENDPOINT,
    api_key=AOAI_KEY,
)

if os.path.exists(PDF_PATH):
    pdf_size_mb = os.path.getsize(PDF_PATH) / 1_048_576
    print(f"PDF found  : {os.path.basename(PDF_PATH)}  ({pdf_size_mb:.1f} MB)")
else:
    print(f"WARNING: PDF not found at {PDF_PATH}")
    print("Please update PDF_PATH in this cell.")
print(f"Cover CSV  : {COVER_CSV}")
print(f"Items CSV  : {LINE_ITEMS_CSV}")
ocr_status = "FOUND (will reuse)" if os.path.exists(OCR_CACHE) else "NOT FOUND (will run OCR)"
print(f"OCR cache  : {ocr_status}")
print(f"Model      : {AOAI_DEPLOYMENT}  (Responses API)")


PDF found  : App 12.pdf  (0.3 MB)
Cover CSV  : C:\Users\KR614XU\Downloads\Ishaan\gc_app12_cover_v3.csv
Items CSV  : C:\Users\KR614XU\Downloads\Ishaan\gc_app12_line_items_v3.csv
OCR cache  : FOUND (will reuse)
Model      : gpt-5.4  (Responses API)


In [31]:
# ── Cell 3 · Render PDF pages → base64 JPEG images (replaces Azure DI OCR) ───
import fitz   # pymupdf
import base64, json

IMAGE_CACHE = os.path.join(BASE_DIR, "gc_app12_page_images.json")
IMAGE_DPI   = 200   # 200 DPI: sharp enough for GPT-5.4 to read dense tables

if os.path.exists(IMAGE_CACHE):
    print(f"Loading cached page images: {os.path.basename(IMAGE_CACHE)}")
    with open(IMAGE_CACHE) as f:
        page_images = {int(k): v for k, v in json.load(f).items()}
    print(f"Loaded {len(page_images)} page images from cache.")
else:
    print(f"Rendering PDF → JPEG images at {IMAGE_DPI} DPI...")
    doc = fitz.open(PDF_PATH)
    page_images = {}
    mat = fitz.Matrix(IMAGE_DPI / 72, IMAGE_DPI / 72)
    for i, page in enumerate(doc):
        pg_num  = i + 1
        pix     = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
        img_bytes = pix.tobytes("jpeg", jpg_quality=88)
        b64       = base64.b64encode(img_bytes).decode("utf-8")
        page_images[pg_num] = b64
        kb = len(img_bytes) / 1024
        print(f"  Page {pg_num}: {kb:,.0f} KB  ({len(b64):,} b64 chars)")
    doc.close()

    with open(IMAGE_CACHE, "w") as f:
        json.dump({str(k): v for k, v in page_images.items()}, f)
    print(f"\nImages cached → {IMAGE_CACHE}")

print(f"\nTotal pages rendered: {len(page_images)}")
for pg_num in sorted(page_images.keys()):
    kb = len(page_images[pg_num]) * 3 / 4 / 1024   # approx raw bytes from b64
    print(f"  pg {pg_num:>3}: ~{kb:,.0f} KB")


Rendering PDF → JPEG images at 200 DPI...
  Page 1: 509 KB  (695,372 b64 chars)
  Page 2: 683 KB  (932,360 b64 chars)
  Page 3: 697 KB  (951,208 b64 chars)
  Page 4: 698 KB  (953,200 b64 chars)
  Page 5: 780 KB  (1,064,536 b64 chars)
  Page 6: 739 KB  (1,009,308 b64 chars)
  Page 7: 690 KB  (942,368 b64 chars)
  Page 8: 264 KB  (361,112 b64 chars)
  Page 9: 816 KB  (1,113,520 b64 chars)
  Page 10: 650 KB  (887,968 b64 chars)
  Page 11: 472 KB  (644,788 b64 chars)
  Page 12: 173 KB  (236,404 b64 chars)

Images cached → C:\Users\KR614XU\Downloads\Ishaan\gc_app12_page_images.json

Total pages rendered: 12
  pg   1: ~509 KB
  pg   2: ~683 KB
  pg   3: ~697 KB
  pg   4: ~698 KB
  pg   5: ~780 KB
  pg   6: ~739 KB
  pg   7: ~690 KB
  pg   8: ~264 KB
  pg   9: ~816 KB
  pg  10: ~650 KB
  pg  11: ~472 KB
  pg  12: ~173 KB


In [35]:
# ── Cell 4 · Unified Extraction — Vision (GPT-5.4 reads PDF page images) ─────
# No Azure Document Intelligence needed — GPT-5.4 sees the actual page layout.
# Strategy: two batches to stay within output token limits
#   Batch A: pages 1-4  → cover_page_sheet + line items for pages 2-4
#   Batch B: pages 5-8  → line items for pages 5-8
# Results are merged; cover comes from Batch A only.
import json, time
try:
    import json_repair
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "json-repair", "--quiet"], check=True)
    import json_repair

MASTER_CACHE = os.path.join(BASE_DIR, "gc_app12_master_extraction.json")

MASTER_PROMPT = """You are an expert construction payment application OCR and data extraction assistant.

Your task is to extract structured data from a JV Payment Application PDF.

The PDF contains:
1. A cover page / AIA G702 Application and Certification for Payment.
2. Continuation sheet pages / AIA G703 schedule of values.

You must extract ONLY:
- Cover page sheet data from the AIA G702 cover page.
- Individual line item data from the AIA G703 continuation sheet.

Do NOT extract:
- Labor backup pages.
- Subtotal rows as output rows.
- Grand total rows as output rows.
- General Conditions backup pages.
- General Requirements backup pages.
- GC/GR detail pages.
- Subcontractor backup pages.
- Invoice backup pages.

GENERAL EXTRACTION RULES:
1. Return clean structured JSON only.
2. Do not include explanations outside the JSON.
3. Do not summarize.
4. Only use values visibly present in the PDF.
5. Do not infer missing values unless specifically instructed.
6. Preserve all individual continuation sheet line items.
7. Do not skip individual line items.
8. Do not merge separate line items.
9. Do not include repeated table headers.
10. Do not include blank rows.
11. Do not output subtotal rows.
12. Do not output grand total rows.
13. Subtotal and grand total rows must be read internally only for validation.
14. Extract numeric values as numbers, not text.
15. Remove dollar signs and commas from currency values.
16. If a value is shown in parentheses, return it as a negative number.
17. If a value is blank, return null.
18. If a numeric cell displays "-", return 0 only when the column is clearly numeric.
19. Percentages must be returned as numeric percentages. Example: "32%" must be returned as 32.
20. If OCR is uncertain, populate the best-read value and add a short note in "Review Notes".
21. Always include the PDF source page number for every extracted line item.

OUTPUT FORMAT:
Return one valid JSON object with exactly these top-level keys:
{ "cover_page_sheet": [], "continuation_sheet": [], "extraction_warnings": [] }

==================================================
SECTION 1: COVER PAGE SHEET EXTRACTION
==================================================

Extract the AIA G702 cover page only (PAGE 1). Return one object inside "cover_page_sheet".
If PAGE 1 is not present in the input, return an empty "cover_page_sheet" array.

Return the cover page object using exactly this JSON structure:
{
  "To Owner": "",
  "From Contractor": "",
  "Project Name": "",
  "Payment Application/Invoice No.": "",
  "Payment Application Period": "",
  "Original Contract Sum": null,
  "Net Change by Change Orders": null,
  "Contract Sum to Date": null,
  "Total Completed & Stored to Date": null,
  "Retainage on Completed Work": null,
  "Retainage on Stored Materials": null,
  "Total Retainage": null,
  "Total Earned Less Retainage": null,
  "Less Previous Certificates for Payment": null,
  "Current Payment Due": null,
  "Balance to Finish Including Retainage": null,
  "Change Order Summary": "",
  "Architect's Signature Present": "",
  "Contractor's Signature Present": "",
  "Source Page": 1,
  "Review Notes": ""
}

COVER PAGE COLUMN MAPPING:
- "To Owner": owner/payor entity listed after "TO OWNER".
- "From Contractor": contractor submitting the application listed after "FROM CONTRACTOR".
- "Project Name": project name or description shown near "PROJECT".
- "Payment Application/Invoice No.": number shown as "APPLICATION NO." or invoice reference.
- "Payment Application Period": period shown as "PERIOD TO".
- "Original Contract Sum": Extract line 1, "ORIGINAL CONTRACT SUM".
- "Net Change by Change Orders": Extract line 2, "Net change by Change Orders".
- "Contract Sum to Date": Extract line 3, "CONTRACT SUM TO DATE".
- "Total Completed & Stored to Date": Extract line 4.
- "Retainage on Completed Work": retainage line 5a amount.
- "Retainage on Stored Materials": retainage line 5b amount.
- "Total Retainage": total retainage below lines 5a and 5b.
- "Total Earned Less Retainage": Extract line 6.
- "Less Previous Certificates for Payment": Extract line 7.
- "Current Payment Due": Extract line 8.
- "Balance to Finish Including Retainage": Extract line 9.
- "Change Order Summary": compact text: "Previous Additions: X; Previous Deductions: Y; Approved This Month Additions: X; Approved This Month Deductions: Y; Total Additions: X; Total Deductions: Y; Net Change: X"
- "Architect's Signature Present": "Yes", "No", or "Unclear".
- "Contractor's Signature Present": "Yes", "No", or "Unclear".
- "Source Page": PDF page number for cover page, usually 1.
- "Review Notes": notes only for uncertainty or unusual formatting.

==================================================
SECTION 2: CONTINUATION SHEET EXTRACTION
==================================================

Extract the AIA G703 continuation sheet table only.
Extract ONLY individual numeric line item rows.
Do NOT output WBS subtotal rows, phase subtotal rows, section subtotal rows, grand total rows, header rows, or blank rows.

For each individual line item row, return one object inside "continuation_sheet":
{
  "Item No.": "",
  "Time Period": "",
  "Phases": "",
  "Type of work": "",
  "Contractor name": "",
  "SCHEDULED ORIGINAL": null,
  "SCHEDULED CHANGE ORDERS": null,
  "SCHEDULED CURRENT": null,
  "WORK COMPLETED FROM PREVIOUS APPLICATION": null,
  "WORK COMPLETED THIS PERIOD": null,
  "MATERIALS PRESENTLY STORED": null,
  "TOTAL COMPLETED AND STORED": null,
  "% (G / C)": null,
  "Balance to Finish (C-G)": null,
  "RETAINAGE (If Variable Rate)": null,
  "Source Page": null,
  "Review Notes": ""
}

CONTINUATION SHEET COLUMN MAPPING:
- "Item No.": line item number, e.g. 001, 002.
- "Time Period": billing period from the application header, e.g. "2/28/2026".
- "Phases": WBS/phase/site grouping for the line item (section header above the item). Examples: P1SI, SITE, TEMP FIRE, FLTL, PTS, PKRD, 8848, CUB, T1-T4, DLO, TEMP FIRE STATION.
- "Type of work": work description from "DESCRIPTION OF WORK". Remove contractor name in parentheses.
- "Contractor name": contractor/subcontractor name shown in parentheses. If none, return null.
- "SCHEDULED ORIGINAL": original scheduled value.
- "SCHEDULED CHANGE ORDERS": change order value.
- "SCHEDULED CURRENT": current scheduled value after change orders.
- "WORK COMPLETED FROM PREVIOUS APPLICATION": cumulative work completed before current period (FIRST dollar amount in D/E column pair).
- "WORK COMPLETED THIS PERIOD": amount billed in THIS period only (SECOND dollar amount in D/E column pair). Do NOT use TOTAL COMPLETED AND STORED for this field.
- "MATERIALS PRESENTLY STORED": stored materials not included in D or E. Often blank/null.
- "TOTAL COMPLETED AND STORED": = D + E + F. ALWAYS a large dollar amount.
- "% (G / C)": completion percentage, 0-100 integer. NEVER a large dollar amount.
- "Balance to Finish (C-G)": = SCHEDULED CURRENT - TOTAL COMPLETED AND STORED.
- "RETAINAGE (If Variable Rate)": retainage withheld for the line item.
- "Source Page": PDF page number where the line item was extracted.
- "Review Notes": only if there is an OCR issue, unclear value, or unusual mapping.

CRITICAL COLUMN ALIGNMENT RULES (vision-based reading):
- Look at the physical column positions in the table image carefully.
- Column D (WORK COMPLETED FROM PREVIOUS APPLICATION) and Column E (WORK COMPLETED THIS PERIOD) are adjacent. When a row has only one filled number in the D/E area and F is blank, that number goes in D and E = null.
- Column F (MATERIALS PRESENTLY STORED) is a separate column between E and G. It is frequently blank/null — do not skip past it.
- Column G (TOTAL COMPLETED AND STORED) is always a large dollar amount equal to D+E+F. Never a 1-3 digit number.
- Column H (% G/C) is always a small 0-100 integer. Never a large dollar amount.
- VERIFY: G = D + E + F before finalising each row.

==================================================
INTERNAL VALIDATION
==================================================

For each WBS grouping: sum individual line items and compare to the PDF subtotal. Correct any errors.
After all items extracted: sum all and compare to PDF GRAND TOTALS. Correct any errors.
Add unresolvable mismatches to "extraction_warnings" only.

==================================================
FINAL OUTPUT RULES
==================================================
Return only valid JSON. No markdown. No code fences. No subtotal rows. No grand total rows.
"""


def parse_master_json(raw):
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
    try:
        result = json.loads(raw)
        if isinstance(result, dict):
            return result
    except json.JSONDecodeError:
        pass
    try:
        result = json_repair.repair_json(raw, return_objects=True)
        if isinstance(result, dict):
            return result
    except Exception:
        pass
    raise ValueError(f"Cannot parse master response: {raw[:200]}")


def call_master_vision(pages, label):
    """Send PDF page images directly to GPT-5.4 — no OCR needed."""
    # Build a single user message with interleaved text labels + images
    # Responses API content types: "input_text" and "input_image"
    msg_content = []
    for pg in pages:
        if pg not in page_images:
            print(f"  [{label}] WARNING: page {pg} image not found, skipping")
            continue
        msg_content.append({"type": "input_text",  "text": f"=== PDF PAGE {pg} ==="})
        msg_content.append({"type": "input_image", "image_url": f"data:image/jpeg;base64,{page_images[pg]}"})

    input_payload = [{"role": "user", "content": msg_content}]

    print(f"  [{label}] pages {min(pages)}-{max(pages)}: {len(pages)} images → GPT-5.4 vision...")
    t0 = time.time()
    resp = aoai_client.responses.create(
        model=AOAI_DEPLOYMENT,
        instructions=MASTER_PROMPT,
        input=input_payload,
        temperature=0,
        max_output_tokens=32000,
    )
    elapsed = time.time() - t0
    raw    = resp.output_text
    result = parse_master_json(raw)
    items  = result.get("continuation_sheet", [])
    cover  = result.get("cover_page_sheet", [])
    print(f"  [{label}] done in {elapsed:.1f}s — {len(cover)} cover fields, {len(items)} line items")
    return result


# ── Load or run extraction ────────────────────────────────────────────────────
if os.path.exists(MASTER_CACHE):
    print(f"Loading cached master extraction: {os.path.basename(MASTER_CACHE)}")
    with open(MASTER_CACHE) as f:
        master_result = json.load(f)
    print("Cache loaded.")
else:
    # Batch A: cover (pg 1) + G703 pages 2-4
    batch_a = call_master_vision(list(range(1, 5)), "Batch A")
    time.sleep(5)
    # Batch B: G703 pages 5-8  (cover page omitted — already got it from Batch A)
    batch_b = call_master_vision(list(range(5, 9)), "Batch B")

    master_result = {
        "cover_page_sheet":    batch_a.get("cover_page_sheet", []),
        "continuation_sheet":  batch_a.get("continuation_sheet", []) + batch_b.get("continuation_sheet", []),
        "extraction_warnings": batch_a.get("extraction_warnings", []) + batch_b.get("extraction_warnings", []),
    }

    with open(MASTER_CACHE, "w", encoding="utf-8") as f:
        json.dump(master_result, f, ensure_ascii=False, indent=2)
    print(f"\nMerged extraction cached → {MASTER_CACHE}")

cover_result      = master_result.get("cover_page_sheet", [{}])[0] if master_result.get("cover_page_sheet") else {}
line_items_result = master_result.get("continuation_sheet", [])
warnings          = master_result.get("extraction_warnings", [])

print(f"\nCover fields populated : {sum(1 for v in cover_result.values() if v not in (None, '', 1))}")
print(f"Line items extracted   : {len(line_items_result)}")
if warnings:
    print(f"\nExtraction warnings ({len(warnings)}):")
    for w in warnings:
        print(f"  ⚠  {w}")
else:
    print("No extraction warnings.")


  [Batch A] pages 1-4: 4 images → GPT-5.4 vision...
  [Batch A] done in 98.5s — 1 cover fields, 102 line items
  [Batch B] pages 5-8: 4 images → GPT-5.4 vision...
  [Batch B] done in 121.4s — 0 cover fields, 131 line items

Merged extraction cached → C:\Users\KR614XU\Downloads\Ishaan\gc_app12_master_extraction.json

Cover fields populated : 19
Line items extracted   : 233

Extraction warnings (7):
  ⚠  Only pages 1-4 were provided; continuation sheet may continue beyond page 4, so extraction may be incomplete.
  ⚠  Some descriptions/contractor names are OCR-uncertain where image text is small or blurred.
  ⚠  Row 096 on page 4 is partially cut off; scheduled/current values are not fully visible.
  ⚠  Several rows labeled GENERAL CONDITIONS/GENERAL REQUIREMENTS were included because they appear as individual G703 line items; backup/detail pages were not provided separately.
  ⚠  Cover page not provided in input pages.
  ⚠  Some rows may contain OCR uncertainty in contractor names or per

In [39]:
# ── Cell 5 · Save Cover Page CSV ─────────────────────────────────────────────
import pandas as pd

app_period = cover_result.get("Payment Application Period", "")

df_cover = pd.DataFrame([cover_result])
df_cover.to_csv(COVER_CSV, index=False)
print(f"Cover CSV saved → {COVER_CSV}")
print()
for k, v in cover_result.items():
    print(f"  {k:<50}: {v}")


Cover CSV saved → C:\Users\KR614XU\Downloads\Ishaan\gc_app12_cover_v2.csv

  To Owner                                          : Boeing Service Company
  From Contractor                                   : BE&K | HITT
  Project Name                                      : Boeing/25/BSC Site Expansion
  Payment Application/Invoice No.                   : 12
  Payment Application Period                        : 2/28/26
  Original Contract Sum                             : 915480436
  Net Change by Change Orders                       : 400427038
  Contract Sum to Date                              : 1315907474
  Total Completed & Stored to Date                  : 422865412
  Retainage on Completed Work                       : 15373232
  Retainage on Stored Materials                     : 71076
  Total Retainage                                   : 15444309
  Total Earned Less Retainage                       : 407421104
  Less Previous Certificates for Payment            : 361844557
  Current

In [ ]:
# ── Cell 6 · Merge pdfplumber numbers + Vision descriptions → v4 CSV ─────────
# Architecture:
#   pdfplumber  → ALL numeric fields  (100% accurate: reads PDF coordinate positions)
#   Vision GPT  → Phases, Type of work, Contractor name  (LLM understands structure)
#   OCR ckpt    → fallback for item 079 (FEE/PTS) which vision skipped
#
# Removes all previous OCR patching / Fix 1 / Fix 2 / Fix 3 hacks.

import pandas as pd
import json
import os

# ── 1. Load pdfplumber numeric data ──────────────────────────────────────────
PLUMBER_CACHE = os.path.join(BASE_DIR, "gc_app12_plumber_nums.json")
with open(PLUMBER_CACHE) as f:
    plumber_data = json.load(f)

plumber_by_ino = {}   # key: zero-padded 3-digit string e.g. "079"
for item in plumber_data["items"]:
    ino = str(item["item_no"]).strip().zfill(3)
    plumber_by_ino[ino] = item

print(f"pdfplumber: {len(plumber_by_ino)} items loaded")

# ── 2. Load Vision descriptions ───────────────────────────────────────────────
vision_by_ino = {}   # key: zero-padded 3-digit string
for item in line_items_result:
    raw_ino = str(item.get("Item No.", "")).strip()
    if raw_ino.upper() in ("GRAND_TOTAL", "GT", ""):
        continue
    ino = raw_ino.lstrip("0").zfill(3) if raw_ino.lstrip("0") else "000"
    vision_by_ino[ino] = item

print(f"Vision: {len(vision_by_ino)} items (item 079 present: {'079' in vision_by_ino})")

# ── 3. Load OCR checkpoint for item 079 fallback ─────────────────────────────
ocr_item79 = None
_OCR_CHECKPOINT = os.path.join(BASE_DIR, "gc_app12_g703_checkpoint.json")
if os.path.exists(_OCR_CHECKPOINT):
    with open(_OCR_CHECKPOINT) as f:
        _ckpt = json.load(f)
    for _pg, _pg_items in _ckpt.get("page_results", {}).items():
        for _item in _pg_items:
            if str(_item.get("item_no", "")).strip().lstrip("0") == "79":
                ocr_item79 = _item
                break
    if ocr_item79:
        print("OCR checkpoint: item 079 found → type='%s', phase='%s'" %
              (ocr_item79.get("type_of_work", ""), ocr_item79.get("phase", "")))

# ── 4. Build merged rows ──────────────────────────────────────────────────────
def safe_int(v, default=0):
    """Convert a float/None to int, use default for None/NaN."""
    if v is None:
        return default
    try:
        return int(round(float(v)))
    except (TypeError, ValueError):
        return default

merged_rows = []
missing_from_vision = []

for ino_padded, pb in sorted(plumber_by_ino.items(), key=lambda x: int(x[0])):
    # Text fields: prefer vision, fall back to OCR checkpoint (item 079), then pdfplumber
    if ino_padded in vision_by_ino:
        vis = vision_by_ino[ino_padded]
        phases         = vis.get("Phases") or ""
        type_of_work   = vis.get("Type of work") or pb.get("description", "")
        contractor_name = vis.get("Contractor name")
        source_page    = vis.get("Source Page") or pb.get("page")
        review_notes   = vis.get("Review Notes")
    elif ino_padded == "079" and ocr_item79:
        phases         = ocr_item79.get("phase") or "PTS"
        type_of_work   = ocr_item79.get("type_of_work") or "FEE"
        contractor_name = ocr_item79.get("contractor_name")
        source_page    = pb.get("page")
        review_notes   = "Text from OCR checkpoint; numbers from pdfplumber"
        missing_from_vision.append(ino_padded)
    else:
        phases         = pb.get("phase", "")
        type_of_work   = pb.get("description", "")
        contractor_name = None
        source_page    = pb.get("page")
        review_notes   = "Text from pdfplumber (not in vision output)"
        missing_from_vision.append(ino_padded)

    # Numeric fields: 100% from pdfplumber
    row = {
        "Item No.":                                  ino_padded,
        "Time Period":                               app_period,
        "Phases":                                    phases,
        "Type of work":                              type_of_work,
        "Contractor name":                           contractor_name,
        "SCHEDULED ORIGINAL":                        safe_int(pb.get("C_ORIG")),
        "SCHEDULED CHANGE ORDERS":                   safe_int(pb.get("C_CO")),
        "SCHEDULED CURRENT":                         safe_int(pb.get("C_CURR")),
        "WORK COMPLETED FROM PREVIOUS APPLICATION":  safe_int(pb.get("D")),
        "WORK COMPLETED THIS PERIOD":                safe_int(pb.get("E")),
        "MATERIALS PRESENTLY STORED":                safe_int(pb.get("F")),
        "TOTAL COMPLETED AND STORED":                safe_int(pb.get("G")),
        "% (G / C)":                                 pb.get("pct", "0%"),
        "Balance to Finish (C-G)":                   safe_int(pb.get("balance")),
        "RETAINAGE (If Variable Rate)":              safe_int(pb.get("retainage")),
        "Source Page":                               source_page,
        "Review Notes":                              review_notes,
    }
    merged_rows.append(row)

if missing_from_vision:
    print(f"Items not in vision output (used fallback): {missing_from_vision}")
print(f"Total merged rows: {len(merged_rows)}")

# ── 4b. Infer contractor name for no-bracket descriptions ────────────────────
# In AIA G703 the Description of Work column encodes two things:
#   • A generic cost-code category (GENERAL CONDITIONS, CCIP, FEE, etc.) — no sub
#   • A sub-contractor name (DABICO AIRPORT SOLUTIONS, COASTAL MILLWORKS, etc.)
# When the description contains parentheses the model already extracts
# "TYPE (CONTRACTOR)". When there are no parentheses the model sets
# contractor_name = None for both cases. The logic below sets
# contractor_name = type_of_work for any row that has no contractor AND
# whose description is not a known generic cost-code category.
GENERIC_COST_CODES = {
    # AIA / construction programme overhead codes — these are NOT company names
    "UNCOMMITTED SUB COST", "GENERAL CONDITIONS", "GENERAL REQUIREMENTS",
    "BUILDERS RISK", "BUILDER'S RISK",
    "BUSINESS LICENSE", "BUSINESS LICENSE/PERMITS", "BUSINESS LICENSE-PERMITS",
    "BUSINESS LICENSE / PERMITS",
    "CCIP", "SDI", "FEE", "PERMITS", "PERMIT",
    "DESIGN ESCALATION", "CONTINGENCY", "FINAL CLEANING",
    "ALLOWANCES", "ALLOWANCE",
    # Some projects add these as line items — still not company names
    "IMP", "PSQIC PROGRAM", "CONTINGENCY ALLOWANCE",
}

inferred_count = 0
for row in merged_rows:
    if row.get("Contractor name"):          # already set — leave it
        continue
    tow = (row.get("Type of work") or "").strip().upper()
    # Skip generic codes and anything that starts with "ALLOWANCE -"
    if tow in GENERIC_COST_CODES or tow.startswith("ALLOWANCE"):
        continue
    # Whatever remains with no contractor IS the subcontractor description
    row["Contractor name"] = row["Type of work"]
    if row.get("Review Notes"):
        row["Review Notes"] += "; contractor inferred from description"
    else:
        row["Review Notes"] = "Contractor inferred from description (no brackets in PDF)"
    inferred_count += 1

if inferred_count:
    print(f"Contractor inference: {inferred_count} rows had contractor name "
          f"inferred from description (no-bracket pattern)")

# ── 5. Save CSV ───────────────────────────────────────────────────────────────
COLUMN_ORDER = [
    "Item No.", "Time Period", "Phases", "Type of work", "Contractor name",
    "SCHEDULED ORIGINAL", "SCHEDULED CHANGE ORDERS", "SCHEDULED CURRENT",
    "WORK COMPLETED FROM PREVIOUS APPLICATION", "WORK COMPLETED THIS PERIOD",
    "MATERIALS PRESENTLY STORED", "TOTAL COMPLETED AND STORED",
    "% (G / C)", "Balance to Finish (C-G)", "RETAINAGE (If Variable Rate)",
    "Source Page", "Review Notes",
]

LINE_ITEMS_CSV_V4 = os.path.join(BASE_DIR, "gc_app12_line_items_v5.csv")
df_items = pd.DataFrame(merged_rows)
for col in COLUMN_ORDER:
    if col not in df_items.columns:
        df_items[col] = None
df_items = df_items[COLUMN_ORDER]
df_items.to_csv(LINE_ITEMS_CSV_V4, index=False)

print(f"\nLine items CSV saved → {LINE_ITEMS_CSV_V4}")
print(f"Rows: {len(df_items)}")

# ── 6. Validation vs PDF Grand Total ─────────────────────────────────────────
PDF_GRAND_TOTAL = {
    "SCHEDULED CURRENT":                        1_315_907_474,
    "WORK COMPLETED FROM PREVIOUS APPLICATION": 375_728_020,
    "WORK COMPLETED THIS PERIOD":               45_715_866,
    "MATERIALS PRESENTLY STORED":               1_421_526,
    "TOTAL COMPLETED AND STORED":               422_865_412,
    "Balance to Finish (C-G)":                  893_042_062,
    "RETAINAGE (If Variable Rate)":             15_444_309,
}

print()
print("Validation — line item sums vs PDF Grand Total:")
print(f"  {'Column':<43}  {'Sum':>16}  {'PDF GT':>16}  {'Diff':>10}  Status")
print("  " + "-" * 100)
all_ok = True
for col, gt in PDF_GRAND_TOTAL.items():
    s    = pd.to_numeric(df_items[col], errors="coerce").sum()
    diff = s - gt
    pct  = abs(diff) / gt * 100 if gt else 0
    flag = f"✓ ({diff:+.0f})" if abs(diff) <= 10 else f"*** {pct:.4f}% off"
    if abs(diff) > 10:
        all_ok = False
    print(f"  {col:<43}  {s:>16,.0f}  {gt:>16,.0f}  {diff:>10,.0f}  {flag}")
print()
if all_ok:
    print("  ✓ ALL columns match PDF Grand Total within $10 rounding (sub-0.001%).")
    print("  ✓ PRODUCTION QUALITY ACHIEVED — 100% numeric accuracy.")
else:
    print("  ✗ Some columns still have discrepancies — investigate pdfplumber output.")


pdfplumber: 234 items loaded
Vision: 233 items (item 079 present: False)
OCR checkpoint: item 079 found → type='FEE', phase='PTS'
Items not in vision output (used fallback): ['079']
Total merged rows: 234
Contractor inference: 3 rows had contractor name inferred from description (no-bracket pattern)


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\KR614XU\\Downloads\\Ishaan\\gc_app12_line_items_v4.csv'